# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and begin analyzing the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` values.

### Dataset Source
The dataset is published as a Croissant schema and accessible from the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the latest version of mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`. We'll work with the Croissant dataset via its URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets and their structure by listing their `@id`s, along with the available fields and columns. All entities are referenced by their `@id`.

In [ ]:
# List available record sets (`@id` and name)
print("Available Record Sets:")
recordset_ids = []
for rs in dataset.record_sets:
    print(f"- @id: {rs.id}")
    recordset_ids.append(rs.id)

# For each record set, list fields and columns by `@id`
for rs in dataset.record_sets:
    print(f"\nRecordSet @id: {rs.id}, name: {rs.name}")
    if rs.fields:
        print("  Fields (by @id):")
        for f in rs.fields:
            print(f"    - {f.id} ({f.name}, dataType: {f.data_type})")
    if rs.columns:
        print("  Columns (by @id):")
        for c in rs.columns:
            print(f"    - {c.id} ({c.name}, dataType: {c.data_type})")

## 3. Data Extraction
We'll load the data from each record set, storing them in a dictionary of DataFrames keyed by their record set `@id`.

*Make sure to use the `@id` values from the overview above for referencing each record set, field, or column you want to analyze.*

In [ ]:
# Extract records from each record set
dataframes = {}
# Use the discovered record_set `@id`s.
print("Loading all record sets into pandas DataFrames...")
for record_set_id in recordset_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}.")
    else:
        print(f"No records available for {record_set_id}.")

# Display columns of the main dataset DataFrame
main_recordset_id = None
for rid, df in dataframes.items():
    if main_recordset_id is None or len(df) > len(dataframes.get(main_recordset_id, pd.DataFrame())):
        main_recordset_id = rid

print(f"\nColumns for main RecordSet (@id={main_recordset_id}):")
print(dataframes[main_recordset_id].columns.tolist())
dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
We perform sample data exploration and processing. We'll select a numeric field (using its `@id`), filter records, normalize the column, and aggregate by a grouping field.

Update `numeric_field_id` and `group_field_id` below to match the actual `@id`s of the numeric and group fields found in section 2 above.

In [ ]:
# --- Choose appropriate `@id`s for your analysis --- #
# Replace these with the correct @id values from your overview above:
numeric_field_id = None  # e.g. 'https://api.app.sen.science/frontiers/7862866/field-age'
group_field_id = None    # e.g. 'https://api.app.sen.science/frontiers/7862866/field-sex'

# For demonstration, guess column names:
df = dataframes[main_recordset_id]
for col in df.columns:
    if numeric_field_id is None and df[col].dtype in [int, float, 'int64', 'float64']:
        numeric_field_id = col
    if group_field_id is None and 'sex' in col.lower():
        group_field_id = col
if numeric_field_id is None or group_field_id is None:
    print("Please inspect the columns and assign numeric_field_id and group_field_id accordingly.")

if numeric_field_id and numeric_field_id in df.columns:
    # Filtering by a threshold
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("A numeric field could not be identified. Please adjust numeric_field_id and group_field_id as appropriate.")

## 5. Visualization
Visualize distribution and group-wise stats for the selected numeric field using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # Boxplot by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Please make sure you've set numeric_field_id and group_field_id to valid column @id values.")

## 6. Conclusion
In this notebook, we loaded and previewed the tabular dataset of cancer survivors with second primary colorectal cancer using the `mlcroissant` library. By referencing all record sets, fields, and columns via their Croissant `@id`, we've demonstrated how to extract, process, and visualize data for reproducible FAIR workflows.

- Dataset metadata and structure are accessible via the Croissant schema.
- Data loading and processing are easily performed by record set or field `@id`.
- You can extend this notebook with further analysis (statistical tests, modeling, etc.), or select additional fields and recordsets as needed.

For more, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/tools/mlcroissant/).